# ============================================================
# **PROJECT TITLE:** Basic Document Q&A Bot using RAG
# **PLATFORM:** Google Colab
# **AUTHOR:** Niladri Dutta
# **PURPOSE:** Build an end-to-end Retrieval Augmented Generation system using PDFs / DOCX / TXT files.
# ============================================================

# ============================================================
# **STEP 1: INSTALL REQUIRED LIBRARIES**
# ============================================================
# **langchain               ->** Main framework for RAG pipeline
# **langchain-community     ->** Loaders / vectorstores
# **langchain-google-genai ->** Gemini integration
# **chromadb                ->** Vector Database
# **sentence-transformers   ->** Embedding model
# **pypdf                   ->** Read PDF files
# **python-docx             ->** Read DOCX files
# **gradio                  ->** Web UI
# ============================================================

In [ ]:
!pip install -qU langchain langchain-community chromadb sentence-transformers pypdf python-docx gradio langchain-google-genai

In [ ]:
!pip install -qU langchain langchain-community langchain-text-splitters

# LangChain's Google Gemini integration package supports Gemini via API key. :contentReference[oaicite:0]{index=0}


# ============================================================
# **STEP 2: IMPORT LIBRARIES**
# ============================================================


In [ ]:
import os
import shutil
import getpass
from google.colab import files

# PDF Loader
from langchain_community.document_loaders import PyPDFLoader, TextLoader

# Correct Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector DB
from langchain_community.vectorstores import Chroma

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# DOCX
import docx

# UI
import gradio as gr

# ============================================================
# **STEP 3: ENTER GEMINI API KEY**
# ============================================================
# Get free API key from:
# https://aistudio.google.com/app/apikey
# ============================================================

In [ ]:
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API Key: ")

# ============================================================
# **STEP 4: UPLOAD DOCUMENTS**
# ============================================================
# Upload 4 to 5 files:
# Example:
# ai.pdf
# ml.pdf
# python.txt
# data_science.docx
# deep_learning.pdf
# ============================================================

In [ ]:
uploaded = files.upload()

# ============================================================
# **STEP 5: CREATE DATA FOLDER**
# ============================================================

In [ ]:
os.makedirs("docs", exist_ok=True)

for file_name in uploaded.keys():
    shutil.move(file_name, f"docs/{file_name}")

print("Files moved to docs folder successfully.")

# ============================================================
# **STEP 6: LOAD ALL DOCUMENTS**
# ============================================================
# This function supports:
# PDF
# TXT
# DOCX
# ============================================================

In [ ]:
documents = []

def load_docx(file_path):
    """
    Custom DOCX loader
    """
    doc = docx.Document(file_path)
    full_text = "\n".join([p.text for p in doc.paragraphs])

    from langchain.schema import Document
    return [Document(
        page_content=full_text,
        metadata={"source": os.path.basename(file_path)}
    )]

for file in os.listdir("docs"):

    path = os.path.join("docs", file)

    if file.endswith(".pdf"):
        loader = PyPDFLoader(path)
        docs = loader.load()
        documents.extend(docs)

    elif file.endswith(".txt"):
        loader = TextLoader(path)
        docs = loader.load()
        documents.extend(docs)

    elif file.endswith(".docx"):
        docs = load_docx(path)
        documents.extend(docs)

print("Total loaded pages/documents:", len(documents))

# ============================================================
# **STEP 7: TEXT CHUNKING**
# ============================================================
# Why chunking?
# Large documents are split into smaller meaningful pieces.
#
# chunk_size=1000:
# Each chunk approx 1000 chars
#
# overlap=200:
# Keeps context continuity between chunks
# ============================================================

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Total chunks created:", len(chunks))

# ============================================================
# **STEP 8: CREATE EMBEDDINGS**
# ============================================================
# Embeddings convert text into vectors.
# Similar meaning texts stay close mathematically.
#
# Using free HuggingFace model: all-MiniLM-L6-v2
# ============================================================

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ============================================================
# **STEP 9: CREATE VECTOR DATABASE**
# ============================================================
# Chroma stores embeddings locally.
# Persisted DB means no need to rebuild every time.
# ============================================================


In [ ]:
db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="chroma_db"
)

db.persist()

print("Vector DB created successfully.")

# ============================================================
# **STEP 10: LOAD GEMINI MODEL**
# ============================================================
# gemini-1.5-flash is fast and cost effective.
# ============================================================

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# ============================================================
# **STEP 11: MAIN QUESTION ANSWERING FUNCTION**
# ============================================================
# Flow:
# User Question
# -> Similarity Search
# -> Retrieve top 3 chunks
# -> Send chunks + question to Gemini
# -> Return grounded answer
# ============================================================

In [ ]:
def ask_bot(question):

    # Search top relevant chunks
    docs = db.similarity_search(question, k=3)

    # If nothing found
    if not docs:
        return "No relevant information found."

    # Prepare context
    context = "\n\n".join([doc.page_content for doc in docs])

    # Collect sources
    sources = []
    for d in docs:
        src = d.metadata.get("source", "Unknown File")
        page = d.metadata.get("page", "N/A")
        sources.append(f"{src} | Page: {page}")

    source_text = "\n".join(list(set(sources)))

    # Prompt engineering
    prompt = f"""
                   You are a helpful Document Q&A assistant.

                   Use ONLY the context below.
                   If answer not present, say:
                   'I could not find this in uploaded documents.'

                   Context:
                   {context}

                  Question:
                  {question}

                  Give a clear professional answer.
                  At the end show source citations.
                  """

    # Generate answer
    response = llm.invoke(prompt)

    final_answer = response.content + "\n\nSources:\n" + source_text

    return final_answer

# ============================================================
# **STEP 12: TEST IN NOTEBOOK**
# ============================================================

In [ ]:
print(ask_bot("What is Resistance?"))

# ============================================================
# **STEP 13: CREATE WEB UI USING GRADIO**
# ============================================================
# Easy web interface
# ============================================================

In [ ]:
interface = gr.Interface(
    fn=ask_bot,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask question from uploaded documents..."
    ),
    outputs="text",
    title="📄 Document Q&A Bot (RAG)",
    description="Upload files, ask questions, get answers with citations."
)

interface.launch(share=True)

# ============================================================
# **END OF PROJECT**
# ============================================================